# PW01 · Source Event Shards

用途：在 Colab 冷启动环境中执行单个 positive_source shard。

边界：仅执行 PW01，不改动主链语义，不扩展到后续 stage。

## 1) 定义路径与参数

本单元定义 PW01 的输入参数，并固定脚本入口路径。

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

NOTEBOOK_NAME = "PW01_Source_Event_Shards"
REPO_URL = "https://github.com/RICHAAARC/CEG-WM.git"
REPO_BRANCH = "organize"
DRIVE_MOUNT_ROOT = Path("/content/drive")
DRIVE_PROJECT_ROOT = DRIVE_MOUNT_ROOT / "MyDrive" / "CEG_WM_PaperWorkflow"
REPO_ROOT = Path("/content/ceg_wm_workspace")
SCRIPT_PATH = REPO_ROOT / "paper_workflow" / "scripts" / "PW01_Source_Event_Shards.py"
CONFIG_PATH = REPO_ROOT / "configs" / "default.yaml"
HF_HOME = REPO_ROOT / "huggingface_cache"
HF_HUB_CACHE = HF_HOME / "hub"
TRANSFORMERS_CACHE = HF_HOME / "transformers"
PERSISTENT_MODEL_CACHE_ROOT = DRIVE_PROJECT_ROOT / "shared_models"
SEMANTIC_MODEL_SOURCE_URLS = {
    "inspyrenet": "https://huggingface.co/plemeri/InSPyReNet/resolve/main/ckpt_base.pth",
}

FAMILY_ID = "paper_eval_family_demo"
SAMPLE_ROLE = "positive_source"
SAMPLE_ROLE_DIRECTORY_NAMES = {
    "positive_source": "positive",
    "clean_negative": "negative",
}
if SAMPLE_ROLE not in SAMPLE_ROLE_DIRECTORY_NAMES:
    raise ValueError(f"unsupported SAMPLE_ROLE: {SAMPLE_ROLE}")
SAMPLE_ROLE_DIRNAME = SAMPLE_ROLE_DIRECTORY_NAMES[SAMPLE_ROLE]
SHARD_INDEX = 0
SHARD_COUNT = 2
STAGE_01_WORKER_COUNT = 2
FORCE_RERUN = False

def run_checked(command, cwd=None):
    print("$", " ".join(str(part) for part in command))
    subprocess.run(command, cwd=cwd, check=True)

def print_json(title, payload):
    print(f"\n[{title}]")
    print(json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True))

## 2) 挂载 Drive 并同步仓库

本单元保证 PW01 可以在空白 Colab 会话中直接冷启动。

In [ ]:
try:
    from google.colab import drive  # type: ignore
    drive.mount(str(DRIVE_MOUNT_ROOT), force_remount=True)
    DRIVE_MOUNT_STATUS = "mounted"
except Exception as exc:
    DRIVE_MOUNT_STATUS = f"skipped: {type(exc).__name__}: {exc}"

DRIVE_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
HF_HOME.mkdir(parents=True, exist_ok=True)
HF_HUB_CACHE.mkdir(parents=True, exist_ok=True)
TRANSFORMERS_CACHE.mkdir(parents=True, exist_ok=True)

if REPO_ROOT.exists() and (REPO_ROOT / ".git").exists():
    origin_result = subprocess.run(
        ["git", "-C", str(REPO_ROOT), "remote", "get-url", "origin"],
        check=False,
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    origin_url = origin_result.stdout.strip()
    if origin_url.rstrip("/") != REPO_URL.rstrip("/"):
        shutil.rmtree(REPO_ROOT)
        run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])
    else:
        run_checked(["git", "fetch", "origin"], cwd=REPO_ROOT)
        run_checked(["git", "checkout", REPO_BRANCH], cwd=REPO_ROOT)
        run_checked(["git", "pull", "origin", REPO_BRANCH], cwd=REPO_ROOT)
else:
    if REPO_ROOT.exists():
        shutil.rmtree(REPO_ROOT)
    run_checked(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_ROOT)])

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print_json(
    "repo_bootstrap",
    {
        "drive_mount_status": DRIVE_MOUNT_STATUS,
        "repo_url": REPO_URL,
        "repo_branch": REPO_BRANCH,
        "repo_root": str(REPO_ROOT),
        "script_path": str(SCRIPT_PATH),
    },
)

## 3) 安装依赖并执行 attestation bootstrap

本单元对齐 01_Paper_Full_Cuda_Parallel.ipynb 的环境准备：安装 transparent-background、准备 InSPyReNet 权重、准备 Hugging Face snapshot，并完成 attestation env bootstrap。

In [ ]:
import os
import shutil
import socket

import torch
from huggingface_hub import HfApi
from scripts.notebook_runtime_common import (
    apply_notebook_model_snapshot_binding,
    load_yaml_mapping,
    write_yaml_mapping,
)
from scripts.workflow_acceptance_common import detect_stage_01_preflight

FAMILY_ROOT = DRIVE_PROJECT_ROOT / "paper_workflow" / "families" / FAMILY_ID
FAMILY_MANIFEST_PATH = FAMILY_ROOT / "manifests" / "paper_eval_family_manifest.json"
SOURCE_EVENT_GRID_PATH = FAMILY_ROOT / "manifests" / "source_event_grid.jsonl"
SOURCE_SHARD_PLAN_PATH = FAMILY_ROOT / "manifests" / "source_shard_plan.json"
SOURCE_SPLIT_PLAN_PATH = FAMILY_ROOT / "manifests" / "source_split_plan.json"

PRECHECK_RESULTS = []

def record_precheck(name, ok, detail):
    PRECHECK_RESULTS.append({
        "name": str(name),
        "ok": bool(ok),
        "detail": str(detail),
    })

record_precheck("Drive 项目根目录存在", DRIVE_PROJECT_ROOT.exists(), str(DRIVE_PROJECT_ROOT))
record_precheck("仓库目录存在", REPO_ROOT.exists(), str(REPO_ROOT))
record_precheck("PW01 脚本存在", SCRIPT_PATH.exists(), str(SCRIPT_PATH))
record_precheck("family 根目录存在", FAMILY_ROOT.exists(), str(FAMILY_ROOT))
record_precheck("family manifest 存在", FAMILY_MANIFEST_PATH.exists(), str(FAMILY_MANIFEST_PATH))
record_precheck("source event grid 存在", SOURCE_EVENT_GRID_PATH.exists(), str(SOURCE_EVENT_GRID_PATH))
record_precheck("source shard plan 存在", SOURCE_SHARD_PLAN_PATH.exists(), str(SOURCE_SHARD_PLAN_PATH))
record_precheck("source split plan 存在", SOURCE_SPLIT_PLAN_PATH.exists(), str(SOURCE_SPLIT_PLAN_PATH))
record_precheck(
    "sample_role 合法",
    SAMPLE_ROLE in SAMPLE_ROLE_DIRECTORY_NAMES,
    f"sample_role={SAMPLE_ROLE}",
)
record_precheck(
    "shard 参数合法",
    isinstance(SHARD_COUNT, int) and SHARD_COUNT > 0 and isinstance(SHARD_INDEX, int) and 0 <= SHARD_INDEX < SHARD_COUNT,
    f"shard_index={SHARD_INDEX}, shard_count={SHARD_COUNT}",
)
record_precheck(
    "STAGE_01_WORKER_COUNT 合法",
    isinstance(STAGE_01_WORKER_COUNT, int) and STAGE_01_WORKER_COUNT in {1, 2},
    f"stage_01_worker_count={STAGE_01_WORKER_COUNT}",
)
record_precheck(
    "MODEL_SNAPSHOT_PATH 已提供",
    isinstance(MODEL_SNAPSHOT_PATH, str) and bool(str(MODEL_SNAPSHOT_PATH).strip()),
    str(MODEL_SNAPSHOT_PATH),
)

family_manifest_obj = {}
manifest_shard_count = None
manifest_sample_roles = []
if FAMILY_MANIFEST_PATH.exists():
    family_manifest_obj = json.loads(FAMILY_MANIFEST_PATH.read_text(encoding="utf-8"))
    source_parameters = family_manifest_obj.get("source_parameters", {})
    if isinstance(source_parameters, dict):
        manifest_shard_count = source_parameters.get("source_shard_count")
    sample_roles_node = family_manifest_obj.get("sample_roles", {})
    if isinstance(sample_roles_node, dict):
        active_roles = sample_roles_node.get("active")
        if isinstance(active_roles, list):
            manifest_sample_roles = [str(item) for item in active_roles if isinstance(item, str)]

record_precheck(
    "manifest 与 SHARD_COUNT 一致",
    isinstance(manifest_shard_count, int) and manifest_shard_count == SHARD_COUNT,
    f"manifest_shard_count={manifest_shard_count}, shard_count={SHARD_COUNT}",
)
record_precheck(
    "manifest 激活当前 sample_role",
    SAMPLE_ROLE in manifest_sample_roles,
    json.dumps(manifest_sample_roles, ensure_ascii=False),
)
record_precheck("FORCE_RERUN 为 bool", isinstance(FORCE_RERUN, bool), str(FORCE_RERUN))
record_precheck(
    "attestation bootstrap 状态可用",
    str(ATTESTATION_BOOTSTRAP.get("status", "")) in {"generated", "reused", "disabled"},
    json.dumps(ATTESTATION_BOOTSTRAP, ensure_ascii=False, sort_keys=True),
)

PRECHECK_RUNTIME_METADATA_ROOT = DRIVE_PROJECT_ROOT / "runtime_state" / NOTEBOOK_NAME / "precheck_runtime_metadata"
PRECHECK_BOUND_CONFIG_PATH = PRECHECK_RUNTIME_METADATA_ROOT / "precheck_bound_config.yaml"
PW01_BOUND_CONFIG_PATH = PRECHECK_BOUND_CONFIG_PATH
PRECHECK_BINDING_ENV = dict(os.environ)
PRECHECK_BINDING_ENV["CEG_WM_MODEL_SNAPSHOT_PATH"] = str(MODEL_SNAPSHOT_PATH)
PRECHECK_BASE_CFG = load_yaml_mapping(CONFIG_PATH)
PRECHECK_BOUND_CFG = apply_notebook_model_snapshot_binding(
    PRECHECK_BASE_CFG,
    env_mapping=PRECHECK_BINDING_ENV,
)
write_yaml_mapping(PW01_BOUND_CONFIG_PATH, PRECHECK_BOUND_CFG)
PRECHECK_MODEL_SOURCE_BINDING = (
    PRECHECK_BOUND_CFG.get("model_source_binding")
    if isinstance(PRECHECK_BOUND_CFG.get("model_source_binding"), dict)
    else None
)
PRECHECK_BINDING_APPLIED = (
    PRECHECK_MODEL_SOURCE_BINDING is not None
    and isinstance(PRECHECK_BOUND_CFG.get("model_snapshot_path"), str)
    and bool(str(PRECHECK_BOUND_CFG.get("model_snapshot_path")).strip())
)

record_precheck(
    "precheck 绑定配置路径",
    PW01_BOUND_CONFIG_PATH.exists(),
    str(PW01_BOUND_CONFIG_PATH),
)
record_precheck(
    "notebook 模型绑定已应用到 precheck 配置",
    PRECHECK_BINDING_APPLIED,
    json.dumps(PRECHECK_MODEL_SOURCE_BINDING, ensure_ascii=False, sort_keys=True)
    if PRECHECK_MODEL_SOURCE_BINDING is not None
    else "<absent>",
)

STAGE_01_PREFLIGHT = detect_stage_01_preflight(PW01_BOUND_CONFIG_PATH)
record_precheck(
    "stage 01 preflight",
    STAGE_01_PREFLIGHT["ok"],
    json.dumps(STAGE_01_PREFLIGHT, ensure_ascii=False, sort_keys=True),
)
record_precheck(
    "CUDA 工具可用",
    STAGE_01_PREFLIGHT["gpu_tool_available"],
    STAGE_01_PREFLIGHT["nvidia_smi_path"],
)
record_precheck(
    "torch.cuda.is_available()",
    torch.cuda.is_available(),
    f"device_count={torch.cuda.device_count() if torch.cuda.is_available() else 0}",
)
record_precheck(
    "attestation env 已就绪",
    not STAGE_01_PREFLIGHT["missing_attestation_env_vars"],
    ", ".join(STAGE_01_PREFLIGHT["missing_attestation_env_vars"]) or "ok",
)
record_precheck(
    "InSPyReNet 权重存在",
    is_valid_weight_file(WEIGHT_PATH),
    f"path={WEIGHT_PATH}, cache_path={PERSISTENT_WEIGHT_PATH}",
)

for module_name in [
    "yaml",
    "huggingface_hub",
    "diffusers",
    "transformers",
    "safetensors",
    "transparent_background",
    "torch",
]:
    try:
        __import__(module_name)
        record_precheck(f"模块可导入: {module_name}", True, "ok")
    except Exception as exc:
        record_precheck(f"模块可导入: {module_name}", False, f"{type(exc).__name__}: {exc}")

hf_model_ok = False
hf_model_detail = "not_checked"
try:
    socket.setdefaulttimeout(15)
    model_info = HfApi().model_info(str(MODEL_IDENTITY["model_id"]))
    hf_model_ok = True
    hf_model_detail = f"accessible: {model_info.id}"
except Exception as exc:
    hf_model_ok = False
    hf_model_detail = f"{type(exc).__name__}: {exc}"
record_precheck("HF 模型可访问", hf_model_ok, hf_model_detail)

disk_usage = shutil.disk_usage(str(REPO_ROOT))
free_gb = disk_usage.free / (1024 ** 3)
record_precheck("磁盘剩余空间", free_gb >= 20.0, f"free={free_gb:.1f} GB，建议 >= 20 GB")

print_json(
    "pw01_precheck",
    {
        "family_id": FAMILY_ID,
        "sample_role": SAMPLE_ROLE,
        "sample_role_dirname": SAMPLE_ROLE_DIRNAME,
        "shard_index": SHARD_INDEX,
        "shard_count": SHARD_COUNT,
        "stage_01_worker_count": STAGE_01_WORKER_COUNT,
        "config_path": str(CONFIG_PATH),
        "precheck_bound_config_path": str(PRECHECK_BOUND_CONFIG_PATH),
        "bound_config_path": str(PW01_BOUND_CONFIG_PATH),
        "model_snapshot_path": str(MODEL_SNAPSHOT_PATH),
        "notebook_model_snapshot_binding_applied": PRECHECK_BINDING_APPLIED,
        "precheck_bound_config": {
            "model_snapshot_path": PRECHECK_BOUND_CFG.get("model_snapshot_path", "<absent>"),
            "model_source_binding": PRECHECK_MODEL_SOURCE_BINDING or "<absent>",
        },
        "items": PRECHECK_RESULTS,
        "stage_01_preflight": STAGE_01_PREFLIGHT,
    },
)

hard_fail = [item for item in PRECHECK_RESULTS if not item["ok"]]
if hard_fail:
    raise RuntimeError(f"PW01 precheck failed: {hard_fail}")

## 4) 运行前 precheck

本单元同时检查 PW01 必要输入，以及与 01_Paper_Full_Cuda_Parallel.ipynb 对齐的 GPU / 模型 / attestation 运行前条件。

In [ ]:
import os
import shutil
import socket

import torch
from huggingface_hub import HfApi
from scripts.notebook_runtime_common import (
    apply_notebook_model_snapshot_binding,
    load_yaml_mapping,
    write_yaml_mapping,
)
from scripts.workflow_acceptance_common import detect_stage_01_preflight

FAMILY_ROOT = DRIVE_PROJECT_ROOT / "paper_workflow" / "families" / FAMILY_ID
FAMILY_MANIFEST_PATH = FAMILY_ROOT / "manifests" / "paper_eval_family_manifest.json"
SOURCE_EVENT_GRID_PATH = FAMILY_ROOT / "manifests" / "source_event_grid.jsonl"
SOURCE_SHARD_PLAN_PATH = FAMILY_ROOT / "manifests" / "source_shard_plan.json"

PRECHECK_RESULTS = []

def record_precheck(name, ok, detail):
    PRECHECK_RESULTS.append({
        "name": str(name),
        "ok": bool(ok),
        "detail": str(detail),
    })

record_precheck("Drive 项目根目录存在", DRIVE_PROJECT_ROOT.exists(), str(DRIVE_PROJECT_ROOT))
record_precheck("仓库目录存在", REPO_ROOT.exists(), str(REPO_ROOT))
record_precheck("PW01 脚本存在", SCRIPT_PATH.exists(), str(SCRIPT_PATH))
record_precheck("family 根目录存在", FAMILY_ROOT.exists(), str(FAMILY_ROOT))
record_precheck("family manifest 存在", FAMILY_MANIFEST_PATH.exists(), str(FAMILY_MANIFEST_PATH))
record_precheck("source event grid 存在", SOURCE_EVENT_GRID_PATH.exists(), str(SOURCE_EVENT_GRID_PATH))
record_precheck("source shard plan 存在", SOURCE_SHARD_PLAN_PATH.exists(), str(SOURCE_SHARD_PLAN_PATH))
record_precheck(
    "shard 参数合法",
    isinstance(SHARD_COUNT, int) and SHARD_COUNT > 0 and isinstance(SHARD_INDEX, int) and 0 <= SHARD_INDEX < SHARD_COUNT,
    f"shard_index={SHARD_INDEX}, shard_count={SHARD_COUNT}",
)
record_precheck(
    "STAGE_01_WORKER_COUNT 合法",
    isinstance(STAGE_01_WORKER_COUNT, int) and STAGE_01_WORKER_COUNT in {1, 2},
    f"stage_01_worker_count={STAGE_01_WORKER_COUNT}",
)
record_precheck(
    "MODEL_SNAPSHOT_PATH 已提供",
    isinstance(MODEL_SNAPSHOT_PATH, str) and bool(str(MODEL_SNAPSHOT_PATH).strip()),
    str(MODEL_SNAPSHOT_PATH),
)

family_manifest_obj = {}
manifest_shard_count = None
if FAMILY_MANIFEST_PATH.exists():
    family_manifest_obj = json.loads(FAMILY_MANIFEST_PATH.read_text(encoding="utf-8"))
    source_parameters = family_manifest_obj.get("source_parameters", {})
    if isinstance(source_parameters, dict):
        manifest_shard_count = source_parameters.get("source_shard_count")

record_precheck(
    "manifest 与 SHARD_COUNT 一致",
    isinstance(manifest_shard_count, int) and manifest_shard_count == SHARD_COUNT,
    f"manifest_shard_count={manifest_shard_count}, shard_count={SHARD_COUNT}",
)
record_precheck("FORCE_RERUN 为 bool", isinstance(FORCE_RERUN, bool), str(FORCE_RERUN))
record_precheck(
    "attestation bootstrap 状态可用",
    str(ATTESTATION_BOOTSTRAP.get("status", "")) in {"generated", "reused", "disabled"},
    json.dumps(ATTESTATION_BOOTSTRAP, ensure_ascii=False, sort_keys=True),
)

PRECHECK_RUNTIME_METADATA_ROOT = DRIVE_PROJECT_ROOT / "runtime_state" / NOTEBOOK_NAME / "precheck_runtime_metadata"
PRECHECK_BOUND_CONFIG_PATH = PRECHECK_RUNTIME_METADATA_ROOT / "precheck_bound_config.yaml"
PW01_BOUND_CONFIG_PATH = PRECHECK_BOUND_CONFIG_PATH
PRECHECK_BINDING_ENV = dict(os.environ)
PRECHECK_BINDING_ENV["CEG_WM_MODEL_SNAPSHOT_PATH"] = str(MODEL_SNAPSHOT_PATH)
PRECHECK_BASE_CFG = load_yaml_mapping(CONFIG_PATH)
PRECHECK_BOUND_CFG = apply_notebook_model_snapshot_binding(
    PRECHECK_BASE_CFG,
    env_mapping=PRECHECK_BINDING_ENV,
)
write_yaml_mapping(PW01_BOUND_CONFIG_PATH, PRECHECK_BOUND_CFG)
PRECHECK_MODEL_SOURCE_BINDING = (
    PRECHECK_BOUND_CFG.get("model_source_binding")
    if isinstance(PRECHECK_BOUND_CFG.get("model_source_binding"), dict)
    else None
)
PRECHECK_BINDING_APPLIED = (
    PRECHECK_MODEL_SOURCE_BINDING is not None
    and isinstance(PRECHECK_BOUND_CFG.get("model_snapshot_path"), str)
    and bool(str(PRECHECK_BOUND_CFG.get("model_snapshot_path")).strip())
)

record_precheck(
    "precheck 绑定配置路径",
    PW01_BOUND_CONFIG_PATH.exists(),
    str(PW01_BOUND_CONFIG_PATH),
)
record_precheck(
    "notebook 模型绑定已应用到 precheck 配置",
    PRECHECK_BINDING_APPLIED,
    json.dumps(PRECHECK_MODEL_SOURCE_BINDING, ensure_ascii=False, sort_keys=True)
    if PRECHECK_MODEL_SOURCE_BINDING is not None
    else "<absent>",
)

STAGE_01_PREFLIGHT = detect_stage_01_preflight(PW01_BOUND_CONFIG_PATH)
record_precheck(
    "stage 01 preflight",
    STAGE_01_PREFLIGHT["ok"],
    json.dumps(STAGE_01_PREFLIGHT, ensure_ascii=False, sort_keys=True),
)
record_precheck(
    "CUDA 工具可用",
    STAGE_01_PREFLIGHT["gpu_tool_available"],
    STAGE_01_PREFLIGHT["nvidia_smi_path"],
)
record_precheck(
    "torch.cuda.is_available()",
    torch.cuda.is_available(),
    f"device_count={torch.cuda.device_count() if torch.cuda.is_available() else 0}",
)
record_precheck(
    "attestation env 已就绪",
    not STAGE_01_PREFLIGHT["missing_attestation_env_vars"],
    ", ".join(STAGE_01_PREFLIGHT["missing_attestation_env_vars"]) or "ok",
)
record_precheck(
    "InSPyReNet 权重存在",
    is_valid_weight_file(WEIGHT_PATH),
    f"path={WEIGHT_PATH}, cache_path={PERSISTENT_WEIGHT_PATH}",
)

nvidia_smi_result = subprocess.run(
    ["nvidia-smi"],
    check=False,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)
nvidia_smi_output = (nvidia_smi_result.stdout or nvidia_smi_result.stderr).splitlines()
record_precheck(
    "nvidia-smi 命令返回 0",
    nvidia_smi_result.returncode == 0,
    " | ".join(nvidia_smi_output[:4]) if nvidia_smi_output else "<no_output>",
)

for module_name in [
    "yaml",
    "huggingface_hub",
    "diffusers",
    "transformers",
    "safetensors",
    "transparent_background",
    "torch",
]:
    try:
        __import__(module_name)
        record_precheck(f"模块可导入: {module_name}", True, "ok")
    except Exception as exc:
        record_precheck(f"模块可导入: {module_name}", False, f"{type(exc).__name__}: {exc}")

hf_model_ok = False
hf_model_detail = "not_checked"
try:
    socket.setdefaulttimeout(15)
    model_info = HfApi().model_info(str(MODEL_IDENTITY["model_id"]))
    hf_model_ok = True
    hf_model_detail = f"accessible: {model_info.id}"
except Exception as exc:
    hf_model_ok = False
    hf_model_detail = f"{type(exc).__name__}: {exc}"
record_precheck("HF 模型可访问", hf_model_ok, hf_model_detail)

disk_usage = shutil.disk_usage(str(REPO_ROOT))
free_gb = disk_usage.free / (1024 ** 3)
record_precheck("磁盘剩余空间", free_gb >= 20.0, f"free={free_gb:.1f} GB，建议 >= 20 GB")

print_json(
    "pw01_precheck",
    {
        "family_id": FAMILY_ID,
        "shard_index": SHARD_INDEX,
        "shard_count": SHARD_COUNT,
        "stage_01_worker_count": STAGE_01_WORKER_COUNT,
        "config_path": str(CONFIG_PATH),
        "precheck_bound_config_path": str(PRECHECK_BOUND_CONFIG_PATH),
        "bound_config_path": str(PW01_BOUND_CONFIG_PATH),
        "model_snapshot_path": str(MODEL_SNAPSHOT_PATH),
        "notebook_model_snapshot_binding_applied": PRECHECK_BINDING_APPLIED,
        "precheck_bound_config": {
            "model_snapshot_path": PRECHECK_BOUND_CFG.get("model_snapshot_path", "<absent>"),
            "model_source_binding": PRECHECK_MODEL_SOURCE_BINDING or "<absent>",
        },
        "items": PRECHECK_RESULTS,
        "stage_01_preflight": STAGE_01_PREFLIGHT,
    },
)

hard_fail = [item for item in PRECHECK_RESULTS if not item["ok"]]
if hard_fail:
    raise RuntimeError(f"PW01 precheck failed: {hard_fail}")

## 5) 执行 PW01 CLI

本单元执行单 shard 任务；若目标 shard 已完成，FORCE_RERUN=True 时会附加 --force-rerun。

In [ ]:
from scripts.gpu_session_peak import build_gpu_peak_display_summary
from scripts.notebook_runtime_common import build_repo_import_subprocess_env

FAMILY_ROOT = DRIVE_PROJECT_ROOT / "paper_workflow" / "families" / FAMILY_ID
SHARD_ROOT = FAMILY_ROOT / "source_shards" / SAMPLE_ROLE_DIRNAME / f"shard_{SHARD_INDEX:04d}"
PW01_SHARD_MANIFEST_PATH = SHARD_ROOT / "shard_manifest.json"
GPU_PEAK_SCRIPT_PATH = REPO_ROOT / "scripts" / "gpu_session_peak.py"
GPU_PEAK_SUMMARY_PATH = SHARD_ROOT / "artifacts" / "gpu_session_peak.json"
COMMAND = [
    sys.executable,
    str(SCRIPT_PATH),
    "--drive-project-root",
    str(DRIVE_PROJECT_ROOT),
    "--family-id",
    FAMILY_ID,
    "--sample-role",
    SAMPLE_ROLE,
    "--shard-index",
    str(SHARD_INDEX),
    "--shard-count",
    str(SHARD_COUNT),
    "--stage-01-worker-count",
    str(STAGE_01_WORKER_COUNT),
    "--bound-config-path",
    str(PW01_BOUND_CONFIG_PATH),
]
if FORCE_RERUN:
    COMMAND.append("--force-rerun")
MONITORED_COMMAND = [
    sys.executable,
    str(GPU_PEAK_SCRIPT_PATH),
    "--output-json",
    str(GPU_PEAK_SUMMARY_PATH),
    "--label",
    NOTEBOOK_NAME,
    "--sample-interval-ms",
    "200",
    "--",
    *COMMAND,
]

NOTEBOOK_SUBPROCESS_ENV = build_repo_import_subprocess_env(repo_root=REPO_ROOT)
NOTEBOOK_SUBPROCESS_ENV["CEG_WM_MODEL_SNAPSHOT_PATH"] = str(MODEL_SNAPSHOT_PATH)
PW01_RESULT = subprocess.run(
    MONITORED_COMMAND,
    cwd=REPO_ROOT,
    env=NOTEBOOK_SUBPROCESS_ENV,
    check=False,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)

if PW01_RESULT.returncode != 0:
    expected_shard_root = FAMILY_ROOT / "source_shards" / SAMPLE_ROLE_DIRNAME / f"shard_{SHARD_INDEX:04d}"
    recent_worker_logs = []
    worker_root = expected_shard_root / "workers"
    if worker_root.exists():
        for log_path in sorted(
            worker_root.rglob("*.log"),
            key=lambda item: item.stat().st_mtime if item.exists() else 0,
            reverse=True,
        )[:6]:
            recent_worker_logs.append(
                {
                    "path": str(log_path),
                    "tail": log_path.read_text(encoding="utf-8", errors="replace").splitlines()[-20:],
                }
            )
    failure_diagnostics = {
        "return_code": PW01_RESULT.returncode,
        "command": COMMAND,
        "monitored_command": MONITORED_COMMAND,
        "stdout_tail": PW01_RESULT.stdout.splitlines()[-40:],
        "stderr_tail": PW01_RESULT.stderr.splitlines()[-40:],
        "sample_role": SAMPLE_ROLE,
        "expected_shard_root": str(expected_shard_root),
        "gpu_session_peak_summary_path": str(GPU_PEAK_SUMMARY_PATH),
        "gpu_session_peak_summary": (
            json.loads(GPU_PEAK_SUMMARY_PATH.read_text(encoding="utf-8"))
            if GPU_PEAK_SUMMARY_PATH.exists() and GPU_PEAK_SUMMARY_PATH.is_file()
            else None
        ),
        "recent_worker_logs": recent_worker_logs,
    }
    print_json("pw01_failure_diagnostics", failure_diagnostics)
    raise RuntimeError(
        "PW01 failed; inspect pw01_failure_diagnostics for command output, GPU peak summary, and worker log tails"
    )

if not PW01_SHARD_MANIFEST_PATH.exists():
    raise FileNotFoundError(
        "PW01 shard manifest missing after successful subprocess execution: "
        f"shard_manifest_path={PW01_SHARD_MANIFEST_PATH} stdout={PW01_RESULT.stdout} stderr={PW01_RESULT.stderr}"
    )

PW01_SUMMARY = json.loads(PW01_SHARD_MANIFEST_PATH.read_text(encoding="utf-8"))
GPU_PEAK_SUMMARY = json.loads(GPU_PEAK_SUMMARY_PATH.read_text(encoding="utf-8"))
GPU_PEAK_NOTEBOOK_SUMMARY = build_gpu_peak_display_summary(GPU_PEAK_SUMMARY)
GPU_PEAK_NOTEBOOK_SUMMARY["summary_path"] = str(GPU_PEAK_SUMMARY_PATH)
print_json("pw01_summary", PW01_SUMMARY)
print_json("gpu_session_peak_summary", GPU_PEAK_NOTEBOOK_SUMMARY)

## 6) 回读 shard manifest

本单元验证 shard_manifest 已生成且状态为 completed。

In [ ]:
SHARD_MANIFEST = PW01_SUMMARY
print_json("pw01_shard_manifest", SHARD_MANIFEST)

if SHARD_MANIFEST.get("status") != "completed":
    raise RuntimeError(f"PW01 shard not completed: {SHARD_MANIFEST.get('status')}")

## 7) 如何扩展并行（显式说明）

扩展规则：
1. 先在 PW00 固定好 source_shard_count。
2. 多个执行器按 shard_index 分片并行执行 PW01。
3. 每个 shard 只写入 source_shards/positive/shard_xxxx，不会互相覆盖。
4. 单个 shard 内可通过 STAGE_01_WORKER_COUNT 控制是单路还是双 worker。
5. 双 worker 的独立日志与结果只写入当前 shard 的 workers/worker_XX。
6. 仅在需要覆盖历史产物时使用 --force-rerun。

In [ ]:
parallel_plan = []
for shard_idx in range(SHARD_COUNT):
    shard_root = FAMILY_ROOT / "source_shards" / SAMPLE_ROLE_DIRNAME / f"shard_{shard_idx:04d}"
    shard_command = [
        sys.executable,
        str(SCRIPT_PATH),
        "--drive-project-root",
        str(DRIVE_PROJECT_ROOT),
        "--family-id",
        FAMILY_ID,
        "--sample-role",
        SAMPLE_ROLE,
        "--shard-index",
        str(shard_idx),
        "--shard-count",
        str(SHARD_COUNT),
        "--stage-01-worker-count",
        str(STAGE_01_WORKER_COUNT),
        "--bound-config-path",
        str(PW01_BOUND_CONFIG_PATH),
    ]
    if FORCE_RERUN:
        shard_command.append("--force-rerun")
    parallel_plan.append(
        {
            "sample_role": SAMPLE_ROLE,
            "shard_index": shard_idx,
            "stage_01_worker_count": STAGE_01_WORKER_COUNT,
            "worker_mode": "single_process" if STAGE_01_WORKER_COUNT == 1 else "shard_local_subprocess_parallel",
            "command": shard_command,
            "bound_config_path": str(PW01_BOUND_CONFIG_PATH),
            "isolation_root": str(shard_root),
            "worker_roots": [
                str(shard_root / "workers" / f"worker_{local_worker_index:02d}")
                for local_worker_index in range(STAGE_01_WORKER_COUNT)
            ],
        }
    )

print_json(
    "parallel_extension_plan",
    {
        "family_id": FAMILY_ID,
        "sample_role": SAMPLE_ROLE,
        "shard_count": SHARD_COUNT,
        "stage_01_worker_count": STAGE_01_WORKER_COUNT,
        "bound_config_path": str(PW01_BOUND_CONFIG_PATH),
        "plan": parallel_plan,
    },
)

## 8) 双路有效性说明

- STAGE_01_WORKER_COUNT = 2 只表示允许在单个 PW01 shard 内启动两个本地 event worker。
- 是否真的形成“双路有效”，取决于当前 shard 实际分到的 event 数是否至少为 2。
- 当前总事件数 = prompt_count × seed_count。
- 当前 shard 分片规则 = event_index % source_shard_count。
- 若希望两个隔离会话都形成有效双路，至少需要满足 total_event_count >= 2 × source_shard_count。
- 每个 worker 只处理当前 shard 内按本地顺序编号切分后的 event 子集；正式 event 落盘仍位于当前 shard 的 events/、records/、artifacts/ 下，worker 自身日志与结果位于 workers/worker_XX/。